# E03 — Grad-CAM Explainability Analysis

Run this notebook only after:

1. E01 has produced the official main checkpoint.
2. E02 has produced the official main test results.

This notebook uses:

```text
outputs/pretrained/main/best_checkpoint.pth
outputs/pretrained/main/test_results/predictions.csv
outputs/pretrained/main/test_results/confused_pairs.csv
```

and saves Grad-CAM outputs under:

```text
outputs/pretrained/main/gradcam_results/
```


## Step 1 — Import dependencies and locate the project directory


In [ ]:
import csv
import json
import os
import sys
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from torch import nn
from torchvision import transforms
from torchvision.models import resnet18


def find_project_root(start_path):
    """Search the current directory and its parents for the project root."""
    current = Path(start_path).resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "src").exists()
            and (candidate / "data" / "processed").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "The project repository could not be found."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

EXPERIMENT_NAME = "main"
EXPERIMENT_ROOT = (
    Path(config.OUTPUT_ROOT)
    / "pretrained"
    / EXPERIMENT_NAME
)

CHECKPOINT_PATH = (
    EXPERIMENT_ROOT
    / "best_checkpoint.pth"
)
TEST_RESULTS_DIR = (
    EXPERIMENT_ROOT
    / "test_results"
)
PREDICTIONS_PATH = (
    TEST_RESULTS_DIR
    / "predictions.csv"
)
CONFUSED_PAIRS_PATH = (
    TEST_RESULTS_DIR
    / "confused_pairs.csv"
)
GRADCAM_OUTPUT_DIR = (
    EXPERIMENT_ROOT
    / "gradcam_results"
)
GRADCAM_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)
print("Experiment:", EXPERIMENT_NAME)
print("Checkpoint:", CHECKPOINT_PATH)
print("Predictions:", PREDICTIONS_PATH)
print("Confused pairs:", CONFUSED_PAIRS_PATH)
print("Grad-CAM output:", GRADCAM_OUTPUT_DIR)


## Step 2 — Validate required E01 and E02 files


In [ ]:
required_files = [
    CHECKPOINT_PATH,
    PREDICTIONS_PATH,
    CONFUSED_PAIRS_PATH,
]

missing_files = [
    path
    for path in required_files
    if not path.exists()
]

if missing_files:
    missing_text = "\n".join(
        str(path)
        for path in missing_files
    )

    raise FileNotFoundError(
        "Required files are missing:\n"
        f"{missing_text}\n\n"
        "Finish E01 main training and E02 main evaluation first."
    )

print("All required files were found.")


## Step 3 — Define model and transforms


In [ ]:
def create_model():
    """Create the ResNet-18 architecture used in E01."""
    model = resnet18(weights=None)
    model.fc = nn.Linear(
        model.fc.in_features,
        config.NUM_CLASSES,
    )
    return model


def create_eval_transform():
    """Create the deterministic model-input transform."""
    image_size = config.IMG_SIZE[0]
    resize_size = int(image_size / 0.875)

    return transforms.Compose(
        [
            transforms.Resize(resize_size),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize(
                config.IMG_MEAN,
                config.IMG_STD,
            ),
        ]
    )


def create_display_transform():
    """Create the geometric transform for displayed images."""
    image_size = config.IMG_SIZE[0]
    resize_size = int(image_size / 0.875)

    return transforms.Compose(
        [
            transforms.Resize(resize_size),
            transforms.CenterCrop(image_size),
        ]
    )


## Step 4 — Load category names and the official main checkpoint


In [ ]:
def load_label_names(manifest_path):
    """Load category names from the training manifest."""
    label_names = {}

    with Path(manifest_path).open(
        "r",
        newline="",
        encoding="utf-8",
    ) as file:
        reader = csv.DictReader(file)

        for row in reader:
            label = int(row["label"])
            label_names[label] = row.get(
                "category_name",
                str(label),
            )

    return label_names


def load_checkpoint_state_dict(checkpoint_path):
    """Load a model state dictionary from a trusted E01 checkpoint."""
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    if (
        isinstance(checkpoint, dict)
        and "model_state_dict" in checkpoint
    ):
        state_dict = checkpoint["model_state_dict"]

    elif (
        isinstance(checkpoint, dict)
        and "state_dict" in checkpoint
    ):
        state_dict = checkpoint["state_dict"]

    elif isinstance(checkpoint, dict):
        state_dict = checkpoint

    else:
        raise TypeError(
            "Unsupported checkpoint format."
        )

    cleaned_state_dict = {}

    for key, value in state_dict.items():
        cleaned_key = key.removeprefix("module.")
        cleaned_state_dict[cleaned_key] = value

    return cleaned_state_dict


LABEL_NAMES = load_label_names(config.TRAIN_CSV)

model = create_model()

state_dict = load_checkpoint_state_dict(
    CHECKPOINT_PATH
)

model.load_state_dict(state_dict)
model = model.to(DEVICE)
model.eval()

print("Category names:", len(LABEL_NAMES))
print("Checkpoint loaded successfully.")
print("Model device:", next(model.parameters()).device)

## Step 5 — Load and validate E02 prediction tables


In [ ]:
predictions_df = pd.read_csv(
    PREDICTIONS_PATH
)
confused_pairs_df = pd.read_csv(
    CONFUSED_PAIRS_PATH
)

required_columns = {
    "file_path",
    "true_label",
    "true_name",
    "pred_label",
    "pred_name",
    "confidence",
    "correct",
    "top5_labels",
    "top5_scores",
}

missing_columns = (
    required_columns
    - set(predictions_df.columns)
)

if missing_columns:
    raise ValueError(
        "predictions.csv is missing columns: "
        + ", ".join(
            sorted(missing_columns)
        )
    )

if predictions_df.empty:
    raise RuntimeError(
        "predictions.csv is empty."
    )

print("Prediction rows:", len(predictions_df))
print(
    "True classes:",
    predictions_df["true_label"].nunique(),
)
print(
    "Predicted classes:",
    predictions_df["pred_label"].nunique(),
)
print(
    "Top-1 accuracy:",
    f"{predictions_df['correct'].mean():.4f}",
)
print(
    "Confused-pair rows:",
    len(confused_pairs_df),
)

display(predictions_df.head())
display(confused_pairs_df.head(10))


## Step 6 — Check for prediction collapse

A warning is displayed if the model predicts too few classes.


In [ ]:
true_class_count = (
    predictions_df["true_label"].nunique()
)
predicted_class_count = (
    predictions_df["pred_label"].nunique()
)

prediction_coverage = (
    predicted_class_count
    / true_class_count
)

print(
    "Prediction class coverage:",
    f"{prediction_coverage:.4f}",
)

if prediction_coverage < 0.20:
    print(
        "Warning: the model predicts fewer than 20% "
        "of the available classes. "
        "Inspect E01 training before interpreting Grad-CAM results."
    )

display(
    predictions_df["pred_label"]
    .value_counts()
    .head(20)
    .rename_axis("pred_label")
    .reset_index(name="prediction_count")
)


## Step 7 — Resolve image paths


In [ ]:
def resolve_image_path(file_path):
    """Resolve an absolute or relative image path from predictions.csv."""
    path = Path(str(file_path))

    if path.is_absolute() and path.exists():
        return path

    candidates = [
        PROJECT_ROOT / path,
        Path(config.DATA_RAW_ROOT) / path,
        PROJECT_ROOT / "data" / "raw" / path,
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError(
        f"Image file not found: {file_path}"
    )


example_path = resolve_image_path(
    predictions_df.iloc[0]["file_path"]
)

print("Example resolved image:", example_path)


## Step 8 — Define Grad-CAM


In [ ]:
class GradCAM:
    """Generate Grad-CAM heatmaps for a selected model layer."""

    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None

        self.forward_handle = (
            target_layer.register_forward_hook(
                self._save_activations
            )
        )

        self.backward_handle = (
            target_layer.register_full_backward_hook(
                self._save_gradients
            )
        )

    def _save_activations(
        self,
        module,
        inputs,
        output,
    ):
        self.activations = output.detach()

    def _save_gradients(
        self,
        module,
        grad_input,
        grad_output,
    ):
        self.gradients = (
            grad_output[0].detach()
        )

    def generate(
        self,
        input_tensor,
        target_class=None,
    ):
        """Generate one normalised Grad-CAM heatmap."""
        self.model.zero_grad(
            set_to_none=True
        )

        logits = self.model(input_tensor)

        if target_class is None:
            target_class = int(
                logits.argmax(dim=1).item()
            )

        target_score = logits[
            :,
            target_class,
        ].sum()

        target_score.backward()

        if (
            self.activations is None
            or self.gradients is None
        ):
            raise RuntimeError(
                "Grad-CAM hooks did not capture data."
            )

        weights = self.gradients.mean(
            dim=(2, 3),
            keepdim=True,
        )

        cam = (
            weights * self.activations
        ).sum(dim=1)

        cam = torch.relu(cam)

        cam = torch.nn.functional.interpolate(
            cam.unsqueeze(1),
            size=input_tensor.shape[-2:],
            mode="bilinear",
            align_corners=False,
        ).squeeze(1)

        cam_min = cam.amin(
            dim=(1, 2),
            keepdim=True,
        )
        cam_max = cam.amax(
            dim=(1, 2),
            keepdim=True,
        )

        cam = (
            cam - cam_min
        ) / (
            cam_max - cam_min + 1e-8
        )

        return (
            cam[0].cpu().numpy(),
            logits.detach(),
        )

    def close(self):
        """Remove registered hooks."""
        self.forward_handle.remove()
        self.backward_handle.remove()


gradcam = GradCAM(
    model=model,
    target_layer=model.layer4[-1],
)

print("Grad-CAM hooks registered.")


## Step 9 — Prepare images and overlays


In [ ]:
EVAL_TRANSFORM = create_eval_transform()
DISPLAY_TRANSFORM = (
    create_display_transform()
)


def prepare_image(image_path):
    """Create model-input and display versions of one image."""
    image = Image.open(
        image_path
    ).convert("RGB")

    input_tensor = (
        EVAL_TRANSFORM(image)
        .unsqueeze(0)
        .to(DEVICE)
    )

    display_image = (
        DISPLAY_TRANSFORM(image)
    )

    display_array = (
        np.asarray(
            display_image,
            dtype=np.float32,
        )
        / 255.0
    )

    return input_tensor, display_array


def create_overlay(
    display_image,
    heatmap,
    alpha=0.45,
):
    """Blend a heatmap with the displayed image."""
    heatmap_rgb = plt.get_cmap(
        "jet"
    )(heatmap)[..., :3]

    overlay = (
        (1.0 - alpha) * display_image
        + alpha * heatmap_rgb
    )

    return np.clip(
        overlay,
        0.0,
        1.0,
    )


## Step 10 — Generate Grad-CAM for one prediction row


In [ ]:
def generate_gradcam_for_row(
    row,
    output_directory,
    include_true_class=True,
):
    """Generate and save Grad-CAM panels for one prediction."""
    image_path = resolve_image_path(
        row["file_path"]
    )

    input_tensor, display_image = (
        prepare_image(image_path)
    )

    true_label = int(row["true_label"])
    pred_label = int(row["pred_label"])
    confidence = float(
        row["confidence"]
    )
    is_correct = bool(
        int(row["correct"])
    )

    predicted_heatmap, _ = (
        gradcam.generate(
            input_tensor,
            target_class=pred_label,
        )
    )

    predicted_overlay = create_overlay(
        display_image,
        predicted_heatmap,
    )

    true_overlay = None

    if (
        include_true_class
        and true_label != pred_label
    ):
        true_heatmap, _ = (
            gradcam.generate(
                input_tensor,
                target_class=true_label,
            )
        )

        true_overlay = create_overlay(
            display_image,
            true_heatmap,
        )

    safe_stem = "".join(
        character
        if (
            character.isalnum()
            or character in "-_"
        )
        else "_"
        for character in image_path.stem
    )

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    figure_path = (
        output_directory
        / (
            f"{safe_stem}"
            f"_true_{true_label}"
            f"_pred_{pred_label}.png"
        )
    )

    panel_count = (
        4
        if true_overlay is not None
        else 3
    )

    plt.figure(
        figsize=(
            18 if panel_count == 4 else 14,
            5,
        )
    )

    plt.subplot(1, panel_count, 1)
    plt.imshow(display_image)
    plt.title("Original image")
    plt.axis("off")

    plt.subplot(1, panel_count, 2)
    plt.imshow(predicted_heatmap)
    plt.title(
        "Predicted-class heatmap\n"
        f"{row['pred_name']} "
        f"({pred_label})"
    )
    plt.axis("off")

    plt.subplot(1, panel_count, 3)
    plt.imshow(predicted_overlay)
    plt.title(
        "Predicted-class overlay\n"
        f"Confidence: {confidence:.4f}"
    )
    plt.axis("off")

    if true_overlay is not None:
        plt.subplot(
            1,
            panel_count,
            4,
        )
        plt.imshow(true_overlay)
        plt.title(
            "True-class overlay\n"
            f"{row['true_name']} "
            f"({true_label})"
        )
        plt.axis("off")

    status = (
        "Correct"
        if is_correct
        else "Incorrect"
    )

    plt.suptitle(
        f"{status} prediction | "
        f"True: {row['true_name']} | "
        f"Predicted: {row['pred_name']}"
    )

    plt.tight_layout()
    plt.savefig(
        figure_path,
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()
    plt.close()

    return {
        "file_path": str(image_path),
        "true_label": true_label,
        "true_name": row["true_name"],
        "pred_label": pred_label,
        "pred_name": row["pred_name"],
        "confidence": confidence,
        "correct": int(is_correct),
        "gradcam_path": str(figure_path),
    }


## Step 11 — Select representative samples

The selection includes:

- High-confidence correct predictions
- High-confidence incorrect predictions
- Examples from the most frequent confused class pairs


In [ ]:
def select_representative_samples(
    predictions,
    confused_pairs,
    correct_count=4,
    incorrect_count=6,
    confused_pair_count=5,
    examples_per_pair=2,
):
    """Select representative rows for explainability analysis."""
    selected_frames = []

    correct_rows = (
        predictions[
            predictions["correct"] == 1
        ]
        .sort_values(
            "confidence",
            ascending=False,
        )
        .head(correct_count)
    )

    incorrect_rows = (
        predictions[
            predictions["correct"] == 0
        ]
        .sort_values(
            "confidence",
            ascending=False,
        )
        .head(incorrect_count)
    )

    selected_frames.extend(
        [
            correct_rows,
            incorrect_rows,
        ]
    )

    for _, pair in (
        confused_pairs
        .head(confused_pair_count)
        .iterrows()
    ):
        pair_rows = predictions[
            (
                predictions["true_label"]
                == int(pair["true_label"])
            )
            & (
                predictions["pred_label"]
                == int(pair["pred_label"])
            )
        ]

        pair_rows = (
            pair_rows
            .sort_values(
                "confidence",
                ascending=False,
            )
            .head(examples_per_pair)
        )

        selected_frames.append(
            pair_rows
        )

    non_empty_frames = [
        frame
        for frame in selected_frames
        if not frame.empty
    ]

    if not non_empty_frames:
        raise RuntimeError(
            "No Grad-CAM samples were selected."
        )

    selected = pd.concat(
        non_empty_frames,
        ignore_index=True,
    )

    selected = selected.drop_duplicates(
        subset=["file_path"],
        keep="first",
    )

    return selected.reset_index(
        drop=True
    )


selected_samples_df = (
    select_representative_samples(
        predictions=predictions_df,
        confused_pairs=confused_pairs_df,
    )
)

print(
    "Selected Grad-CAM samples:",
    len(selected_samples_df),
)

display(selected_samples_df)


## Step 12 — Generate Grad-CAM visualisations

This cell is enabled by default and normally generates approximately 20 figures.


In [ ]:
RUN_GRADCAM = True

if RUN_GRADCAM:
    gradcam_rows = []

    for index, row in (
        selected_samples_df.iterrows()
    ):
        print(
            f"Generating Grad-CAM "
            f"{index + 1}/"
            f"{len(selected_samples_df)}"
        )

        result = (
            generate_gradcam_for_row(
                row=row,
                output_directory=(
                    GRADCAM_OUTPUT_DIR
                ),
                include_true_class=True,
            )
        )

        gradcam_rows.append(result)

    gradcam_summary_df = pd.DataFrame(
        gradcam_rows
    )

    summary_path = (
        GRADCAM_OUTPUT_DIR
        / "gradcam_summary.csv"
    )

    gradcam_summary_df.to_csv(
        summary_path,
        index=False,
    )

    print(
        "\nGrad-CAM generation completed."
    )
    print("Summary file:", summary_path)

    display(gradcam_summary_df)


## Step 13 — Optional manual analysis


In [ ]:
RUN_SINGLE_IMAGE_ANALYSIS = False
ROW_INDEX = 0

if RUN_SINGLE_IMAGE_ANALYSIS:
    if not (
        0
        <= ROW_INDEX
        < len(predictions_df)
    ):
        raise IndexError(
            "ROW_INDEX is outside the valid range."
        )

    manual_output_dir = (
        GRADCAM_OUTPUT_DIR
        / "manual_examples"
    )

    result = generate_gradcam_for_row(
        row=predictions_df.iloc[
            ROW_INDEX
        ],
        output_directory=(
            manual_output_dir
        ),
        include_true_class=True,
    )

    display(pd.DataFrame([result]))


## Grad-CAM Analysis and Findings

### Correct predictions

For the high-confidence correct predictions, the Grad-CAM maps generally
focused on meaningful parts of the organism. For example, the activation maps
for the owl, pelican, echidna, and flowering plant were concentrated around
the visible body, head, or central plant structure.

This suggests that the model often relied on relevant organism-level visual
features rather than using only the surrounding background. These examples
also show that the progressively fine-tuned ResNet-18 learned spatially
meaningful representations for several different taxonomic groups.

### Fine-grained classification errors

Some incorrect predictions still produced activation maps centred on the
organism. For example, the butterfly was confused with another visually
similar butterfly species, and the beetle was confused with another species
from the same genus. In these cases, the model detected the correct object
region but failed to distinguish subtle species-level features.

This indicates that some errors were caused by genuine fine-grained visual
similarity rather than complete failure to locate the organism.

### Weak and diffuse activation maps

Several incorrectly classified plant, reptile, and bird examples produced
weak, diffuse, or nearly uniform Grad-CAM maps. These examples suggest that
the model did not have a strongly localised positive feature supporting the
selected class.

Possible reasons include small organisms, cluttered backgrounds, visually
similar classes, limited image resolution, and insufficiently distinctive
high-level features. In some cases, contextual features such as vegetation,
branches, or surrounding texture may also have influenced the prediction.

### Confusable species pairs

The common error pairs included visually similar species such as:

- `Pinus resinosa` and `Pinus virginiana`
- `Phalaris arundinacea` and `Calamagrostis canadensis`
- `Urosaurus ornatus` and `Urosaurus nigricaudus`
- `Amazona finschi` and `Amazona albifrons`
- `Lethe eurydice` and `Lethe appalachia`

These pairs share similar shapes, colours, textures, or taxonomic
characteristics. The Grad-CAM results suggest that the network often attended
to the correct general object region but did not consistently capture the
small local features needed to separate closely related species.

### Overall interpretation

The Grad-CAM analysis shows that the model generally learned meaningful
organism-level features for confident correct predictions. However, its
performance was limited by fine-grained similarity, background clutter, and
weak localisation in difficult cases.

Future improvements could include higher-resolution inputs, stronger
fine-grained feature extraction, attention-based architectures, object
cropping or segmentation, and more training examples per species.